In [24]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import BCEDiceLoss, BereaPatchDataset, FiLMRoutedUNet3D, dice_score_from_logits
from utils.training import EarlyStopping, MetricTracker

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct device: cuda


In [25]:
CUBE_SIZE = 128
BATCH_SIZE = 1
EPOCHS = 10
LR = 1e-4
PATIENCE = 3
MIN_DELTA = 1e-4
BASE_CHANNELS = 16
CTX_DIM = 64
CHECKPOINT_PATH = MODEL_DIR / "film_routed_unet3d_best.pth"


In [26]:
train_ds = BereaPatchDataset(ROOT, split="train", cube_size=CUBE_SIZE, use_raw_gray=False)
val_ds = BereaPatchDataset(ROOT, split="val", cube_size=CUBE_SIZE, use_raw_gray=False, noise_types=["none"])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print("train:", len(train_ds), "val:", len(val_ds))


train: 265 val: 78


In [27]:
model = FiLMRoutedUNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=BASE_CHANNELS,
    ctx_dim=CTX_DIM,
).to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
print("parameters:", sum(p.numel() for p in model.parameters()))


parameters: 1466145


In [28]:
early_stopping = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA, mode="min")
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_stats = MetricTracker()
    train_bar = tqdm(train_loader, desc=f"train {epoch}/{EPOCHS}", leave=False)
    for batch in train_bar:
        x = batch["x"].to(device)
        y = batch["y"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(x)
        loss, bce_loss, dice_loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        with torch.no_grad():
            dice = dice_score_from_logits(logits, y)
        train_stats.update("loss", float(loss.detach().cpu()), batch_size)
        train_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
        train_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
        train_stats.update("dice", float(dice.detach().cpu()), batch_size)
        train_bar.set_postfix(train_stats.postfix("loss", "bce", "dice_loss", "dice"))

    model.eval()
    val_stats = MetricTracker()
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"val {epoch}/{EPOCHS}", leave=False)
        for batch in val_bar:
            x = batch["x"].to(device)
            y = batch["y"].to(device)
            logits, _ = model(x)
            loss, bce_loss, dice_loss = criterion(logits, y)
            dice = dice_score_from_logits(logits, y)

            batch_size = x.size(0)
            val_stats.update("loss", float(loss.detach().cpu()), batch_size)
            val_stats.update("bce", float(bce_loss.detach().cpu()), batch_size)
            val_stats.update("dice_loss", float(dice_loss.detach().cpu()), batch_size)
            val_stats.update("dice", float(dice.detach().cpu()), batch_size)
            val_bar.set_postfix(val_stats.postfix("loss", "bce", "dice_loss", "dice"))

    train_metrics = train_stats.as_dict()
    val_metrics = val_stats.as_dict()
    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_dice": val_metrics["dice"],
    })

    print(
        f"epoch={epoch} "
        f"train_loss={train_metrics['loss']:.4f} train_dice={train_metrics['dice']:.4f} "
        f"val_loss={val_metrics['loss']:.4f} val_dice={val_metrics['dice']:.4f}"
    )
    print("alpha:", model.router.alpha().detach().cpu())

    improved = early_stopping.step(val_metrics["loss"], epoch=epoch)
    if improved:
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch,
            "val_loss": val_metrics["loss"],
            "val_dice": val_metrics["dice"],
            "history": history,
            "base_channels": BASE_CHANNELS,
            "ctx_dim": CTX_DIM,
        }, CHECKPOINT_PATH)
        print("сохранено:", CHECKPOINT_PATH)
    else:
        print(f"ранняя остановка: {early_stopping.bad_epochs}/{PATIENCE} эпох без улучшения val_loss")

    if early_stopping.should_stop:
        print(f"обучение остановлено на эпохе {epoch}; лучший val_loss={early_stopping.best:.4f}")
        break


epoch=1 train_loss=0.2868 train_dice=0.9193 val_loss=0.1459 val_dice=0.9595
alpha: tensor([[0.9477, 0.0174, 0.0174, 0.0174],
        [0.0174, 0.9477, 0.0175, 0.0174],
        [0.0173, 0.0173, 0.9481, 0.0173],
        [0.0186, 0.0174, 0.0185, 0.9455]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=2 train_loss=0.0781 train_dice=0.9551 val_loss=0.0618 val_dice=0.9656
alpha: tensor([[0.9472, 0.0176, 0.0176, 0.0176],
        [0.0175, 0.9476, 0.0175, 0.0175],
        [0.0173, 0.0173, 0.9481, 0.0173],
        [0.0196, 0.0174, 0.0195, 0.9435]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=3 train_loss=0.0527 train_dice=0.9583 val_loss=0.0441 val_dice=0.9704
alpha: tensor([[0.9469, 0.0177, 0.0177, 0.0177],
        [0.0175, 0.9475, 0.0175, 0.0175],
        [0.0173, 0.0173, 0.9480, 0.0173],
        [0.0202, 0.0175, 0.0201, 0.9422]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=4 train_loss=0.0467 train_dice=0.9608 val_loss=0.0447 val_dice=0.9648
alpha: tensor([[0.9466, 0.0178, 0.0178, 0.0178],
        [0.0175, 0.9475, 0.0175, 0.0175],
        [0.0173, 0.0173, 0.9480, 0.0173],
        [0.0206, 0.0176, 0.0205, 0.9413]])
ранняя остановка: 1/3 эпох без улучшения val_loss


epoch=5 train_loss=0.0441 train_dice=0.9621 val_loss=0.0376 val_dice=0.9711
alpha: tensor([[0.9465, 0.0179, 0.0178, 0.0179],
        [0.0175, 0.9476, 0.0175, 0.0175],
        [0.0173, 0.0174, 0.9480, 0.0173],
        [0.0209, 0.0177, 0.0208, 0.9406]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=6 train_loss=0.0427 train_dice=0.9630 val_loss=0.0339 val_dice=0.9744
alpha: tensor([[0.9463, 0.0179, 0.0178, 0.0179],
        [0.0175, 0.9476, 0.0175, 0.0175],
        [0.0173, 0.0174, 0.9479, 0.0174],
        [0.0212, 0.0177, 0.0211, 0.9399]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=7 train_loss=0.0422 train_dice=0.9635 val_loss=0.0364 val_dice=0.9704
alpha: tensor([[0.9464, 0.0179, 0.0178, 0.0179],
        [0.0174, 0.9477, 0.0174, 0.0174],
        [0.0173, 0.0174, 0.9479, 0.0174],
        [0.0215, 0.0178, 0.0213, 0.9394]])
ранняя остановка: 1/3 эпох без улучшения val_loss


epoch=8 train_loss=0.0406 train_dice=0.9646 val_loss=0.0452 val_dice=0.9601
alpha: tensor([[0.9462, 0.0180, 0.0179, 0.0180],
        [0.0174, 0.9477, 0.0175, 0.0174],
        [0.0173, 0.0174, 0.9479, 0.0174],
        [0.0218, 0.0179, 0.0215, 0.9389]])
ранняя остановка: 2/3 эпох без улучшения val_loss


epoch=9 train_loss=0.0373 train_dice=0.9679 val_loss=0.0278 val_dice=0.9784
alpha: tensor([[0.9460, 0.0180, 0.0179, 0.0180],
        [0.0174, 0.9476, 0.0175, 0.0174],
        [0.0173, 0.0174, 0.9479, 0.0174],
        [0.0221, 0.0179, 0.0218, 0.9382]])
сохранено: c:\Users\F\Desktop\ДЗ\ОАИП\micro_ct\models\film_routed_unet3d_best.pth


epoch=10 train_loss=0.0367 train_dice=0.9681 val_loss=0.0300 val_dice=0.9755
alpha: tensor([[0.9458, 0.0181, 0.0180, 0.0181],
        [0.0174, 0.9476, 0.0175, 0.0175],
        [0.0173, 0.0174, 0.9479, 0.0174],
        [0.0223, 0.0180, 0.0221, 0.9376]])
ранняя остановка: 1/3 эпох без улучшения val_loss
